In [1]:
import pickle
import re
from pathlib import Path

import numpy as np
import pandas as pd
from rank_bm25 import BM25Okapi

from sentence_transformers import SentenceTransformer
import nltk
from nltk.corpus import stopwords

In [2]:
BASE_DIR = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd()

DOCS_PATH = BASE_DIR / "data" / "processed" / "documents.parquet"
BM25_PATH = BASE_DIR / "models" / "bm25_model.pkl"
EMBEDDINGS_PATH = BASE_DIR / "data" / "processed" / "embeddings.npy"

In [3]:
nltk.download("stopwords", quiet=True)
stop_words = set(stopwords.words("english"))

def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    tokens = text.split()
    tokens = [word for word in tokens if word not in stop_words]
    return tokens

In [4]:
docs = pd.read_parquet(DOCS_PATH)

with open(BM25_PATH, "rb") as f:
    bm25 = pickle.load(f)

docs.head()

,title,text,rating,product_title
0,Five Stars,As described & fast ship!,5.0,DancingNail 18 Color Rolls Striping Tape Line ...
1,great quality product,I have been looking at the name brand wet skin...,5.0,"100% Natural, Organic, Vegan Wet Skin Moisturi..."
2,works great for a reasonable price,I use this to remove eye makeup. Especially ma...,5.0,bemndful Plant-based Micellar Cleansing Water ...
3,These colors are great!,I really liked this palette a lot. The colors ...,4.0,essence | Welcome to London Eyeshadow Palette ...
4,It's working great so far!,I have been using GrandeLash for over a year n...,5.0,Osmotics Eye Lash Enhancing Serum - Eyelash & ...


In [5]:
docs["semantic_text"] = (
    docs["product_title"].fillna("").astype(str) + " " +
    docs["title"].fillna("").astype(str) + " " +
    docs["text"].fillna("").astype(str)
)

docs["semantic_text"].head()

0    DancingNail 18 Color Rolls Striping Tape Line ...
1    100% Natural, Organic, Vegan Wet Skin Moisturi...
2    bemndful Plant-based Micellar Cleansing Water ...
3    essence | Welcome to London Eyeshadow Palette ...
4    Osmotics Eye Lash Enhancing Serum - Eyelash & ...
Name: semantic_text, dtype: str

In [6]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
if EMBEDDINGS_PATH.exists():
    print("Loading saved embeddings...")
    doc_embeddings = np.load(EMBEDDINGS_PATH)
else:
    print("Computing embeddings...")
    doc_embeddings = model.encode(
        docs["semantic_text"].tolist(),
        show_progress_bar=True,
        normalize_embeddings=True
    )
    np.save(EMBEDDINGS_PATH, doc_embeddings)

doc_embeddings.shape

Loading saved embeddings...


(701528, 384)

In [11]:
def search_bm25(query, top_k=3):
    query_tokens = preprocess_text(query)
    scores = np.array(bm25.get_scores(query_tokens))

    top_idx = np.argsort(-scores)[:top_k]

    results = docs.iloc[top_idx].copy()
    results["bm25_score"] = scores[top_idx]

    cols = [c for c in ["title", "text", "rating", "bm25_score"] if c in results.columns]
    return results[cols].reset_index(drop=True)

In [12]:
search_bm25("good moisturizer for dry skin", top_k=3)

,title,text,rating,bm25_score
0,Goes on smoothly and lightweight,Good moisturizer for my very dry skin for dayt...,5.0,18.898853
1,Very good moisturizer.,Very Good moisturizer. I use it on my legs for...,4.0,17.509216
2,Not good for people with dry skin,Not good for people with dry skin. Doesn't moi...,4.0,17.227494


In [13]:
def search_semantic(query, top_k=5):
    query_embedding = model.encode([query], normalize_embeddings=True)[0]
    scores = doc_embeddings @ query_embedding

    top_idx = np.argsort(-scores)[:top_k]

    results = docs.iloc[top_idx].copy()
    results["semantic_score"] = scores[top_idx]

    cols = [c for c in ["title", "text", "rating", "semantic_score"] if c in results.columns]
    return results[cols].reset_index(drop=True)

In [14]:
search_semantic("good moisturizer for dry skin", top_k=3)

,title,text,rating,semantic_score
0,I like it.,I've been using this for a couple months now. ...,4.0,0.879136
1,Very refreshing,Feels like ice water splashing on your face. W...,5.0,0.851046
2,Amazing,This wasn’t for me but it was at the same time...,5.0,0.849029


In [15]:
def min_max_scale(x):
    x = np.asarray(x, dtype=float)
    denom = x.max() - x.min()
    if denom == 0:
        return np.zeros_like(x)
    return (x - x.min()) / denom

In [16]:
def search_hybrid(query, top_k=5, alpha=0.5):
    # BM25 scores
    query_tokens = preprocess_text(query)
    bm25_scores = np.array(bm25.get_scores(query_tokens), dtype=float)

    # Semantic scores
    query_embedding = model.encode([query], normalize_embeddings=True)[0]
    semantic_scores = doc_embeddings @ query_embedding

    # Normalize before combining
    bm25_scaled = min_max_scale(bm25_scores)
    semantic_scaled = min_max_scale(semantic_scores)

    hybrid_scores = alpha * bm25_scaled + (1 - alpha) * semantic_scaled

    top_idx = np.argsort(-hybrid_scores)[:top_k]

    results = docs.iloc[top_idx].copy()
    results["bm25_score"] = bm25_scores[top_idx]
    results["semantic_score"] = semantic_scores[top_idx]
    results["hybrid_score"] = hybrid_scores[top_idx]

    cols = [c for c in [
        "title", "text", "rating","hybrid_score"
    ] if c in results.columns]

    return results[cols].reset_index(drop=True)

In [17]:
search_hybrid("good moisturizer for dry skin", top_k=3, alpha=0.5)

,title,text,rating,hybrid_score
0,Too oily,This product is good as skin moisturizer but t...,3.0,0.750736
1,Good moisturizer at a good price,Good moisturizer that soaks in quickly. Not to...,4.0,0.684180
2,Good for normal/oily skin. Scent is very clean,This toner smells so clean and nice. I wouldn'...,5.0,0.675828


In [18]:
query = "good moisturizer for dry skin"

print("BM25")
display(search_bm25(query, top_k=3))

print("Semantic")
display(search_semantic(query, top_k=3))

print("Hybrid")
display(search_hybrid(query, top_k=3, alpha=0.5))

BM25


,title,text,rating,bm25_score
0,Goes on smoothly and lightweight,Good moisturizer for my very dry skin for dayt...,5.0,18.898853
1,Very good moisturizer.,Very Good moisturizer. I use it on my legs for...,4.0,17.509216
2,Not good for people with dry skin,Not good for people with dry skin. Doesn't moi...,4.0,17.227494


Semantic


,title,text,rating,semantic_score
0,I like it.,I've been using this for a couple months now. ...,4.0,0.879136
1,Very refreshing,Feels like ice water splashing on your face. W...,5.0,0.851046
2,Amazing,This wasn’t for me but it was at the same time...,5.0,0.849029


Hybrid


,title,text,rating,hybrid_score
0,Too oily,This product is good as skin moisturizer but t...,3.0,0.750736
1,Good moisturizer at a good price,Good moisturizer that soaks in quickly. Not to...,4.0,0.684180
2,Good for normal/oily skin. Scent is very clean,This toner smells so clean and nice. I wouldn'...,5.0,0.675828
